# World Events MCP Server — News Intelligence Colab Notebook

Real-time news intelligence powered by free public APIs — no API keys required.

| # | Tool | Description |
|---|---|---|
| 4 | `intel_news_feed` | RSS aggregation — 119 feeds across 24 categories |
| 5 | `intel_gdelt_timeline` | Coverage volume timeline — dual-query comparison chart |
| 6 | `intel_gdelt_timeline` | Tone timeline — sentiment shift chart |
| 7 | `intel_status` | Server health and data source status |
| 8 | `intel_gdelt_search` | Custom GDELT query (artlist, timeline, tone, tonechart) |

## 1. Install

In [ ]:
# Clone the repo and install the package
!git clone https://github.com/dshipley71/WorldEventsPipeline.git 2>/dev/null || echo 'Already cloned'
!pip install -q -e '/content/WorldEventsPipeline/world-events-mcp'

## 2. MCP Client Helper

The server speaks **JSON-RPC 2.0 over stdio**. This helper manages the subprocess and handles the protocol handshake.

In [ ]:
import json
import subprocess
import threading
import queue
import time
import sys
from IPython.display import display, JSON


class MCPClient:
    """Minimal synchronous MCP client over stdio."""

    def __init__(self):
        self._proc = None
        self._reader_thread = None
        self._responses: queue.Queue = queue.Queue()
        self._stderr_lines: list[str] = []
        self._id = 0

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    def connect(self):
        """Start the MCP server subprocess and run the initialize handshake."""
        self._proc = subprocess.Popen(
            ["world-events-mcp"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )
        # Background thread: read stdout lines and push to response queue
        self._reader_thread = threading.Thread(
            target=self._read_loop, daemon=True
        )
        self._reader_thread.start()

        # MCP initialize handshake
        resp = self._send("initialize", {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "colab-tester", "version": "1.0"},
        })
        # Acknowledge initialization
        self._notify("notifications/initialized", {})
        print(f"✅ Connected — server: {resp.get('result', {}).get('serverInfo', {})}")
        return self

    def disconnect(self):
        """Terminate the server process."""
        if self._proc:
            self._proc.terminate()
            self._proc = None
        print("Disconnected.")

    def __enter__(self):
        return self.connect()

    def __exit__(self, *_):
        self.disconnect()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def list_tools(self) -> list[dict]:
        """Return all available tool definitions."""
        resp = self._send("tools/list", {})
        return resp.get("result", {}).get("tools", [])

    def call_tool(self, name: str, arguments: dict | None = None, timeout: float = 30.0) -> dict:
        """Invoke a named tool with the given arguments."""
        resp = self._send(
            "tools/call",
            {"name": name, "arguments": arguments or {}},
            timeout=timeout,
        )
        result = resp.get("result", {})
        # Unwrap MCP content envelope
        contents = result.get("content", [])
        for block in contents:
            if block.get("type") == "text":
                try:
                    return json.loads(block["text"])
                except json.JSONDecodeError:
                    return {"raw": block["text"]}
        return result

    # ------------------------------------------------------------------
    # Internal
    # ------------------------------------------------------------------

    def _next_id(self) -> int:
        self._id += 1
        return self._id

    def _write(self, msg: dict):
        line = json.dumps(msg) + "\n"
        self._proc.stdin.write(line)
        self._proc.stdin.flush()

    def _send(self, method: str, params: dict, timeout: float = 30.0) -> dict:
        msg_id = self._next_id()
        self._write({"jsonrpc": "2.0", "id": msg_id, "method": method, "params": params})
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                resp = self._responses.get(timeout=1.0)
                if resp.get("id") == msg_id:
                    return resp
                # put back if it's for another id
                self._responses.put(resp)
            except queue.Empty:
                pass
        raise TimeoutError(f"No response for method={method!r} within {timeout}s")

    def _notify(self, method: str, params: dict):
        """Send a notification (no id, no response expected)."""
        self._write({"jsonrpc": "2.0", "method": method, "params": params})

    def _read_loop(self):
        for line in self._proc.stdout:
            line = line.strip()
            if not line:
                continue
            try:
                msg = json.loads(line)
                if "id" in msg:
                    self._responses.put(msg)
                # drop server-sent notifications
            except json.JSONDecodeError:
                pass


# Instantiate and connect once for the whole notebook
client = MCPClient()
client.connect()

## 3. List Available Tools

In [ ]:
tools = client.list_tools()
print(f"Total tools: {len(tools)}\n")
for t in tools:
    print(f"  {t['name']:<40} {t.get('description', '')[:70]}")

## 4. News — Feed with Category Filters

In [ ]:
result = client.call_tool("intel_news_feed", {"categories": ["geopolitics", "military"], "limit": 15})

articles = result.get("articles", [])
print(f"Latest {len(articles)} articles:\n")
for a in articles:
    print(f"  [{a.get('category',''):<12}] {a.get('title','')[:80]}")
    print(f"    {a.get('published','')[:19]}  {a.get('source','')}")
    print()

## 5. GDELT Timeline — Coverage Volume Over Time

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# ── Parameters ───────────────────────────────────────────────────────────────
QUERY1    = "ukraine war"
QUERY2    = "taiwan strait"    # set to None to plot a single topic
MODE      = "timelinevol"      # timelinevol | timelinevolraw | timelinetone |
                                # timelinesourcecountry | timelinelang
TIMESPAN  = "7d"               # 15min | 6h | 7d | 2w | 1m | 3m
SMOOTHING = 3                  # 1–30  (higher = smoother curve)
# ─────────────────────────────────────────────────────────────────────────────

# GDELT date formats (try in order — API uses YYYYMMDDHHMMSS without separators)
_GDELT_DATE_FMTS = ["%Y%m%dT%H%M%S", "%Y%m%d%H%M%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"]

def _parse_series(data):
    """Extract (dates, values) from a GDELT timeline data array."""
    dates, values = [], []
    for point in data:
        if not isinstance(point, dict):
            continue
        raw_date = point.get("date") or point.get("Date") or point.get("datetime")
        # Use .get with explicit default to avoid 0.0 being treated as falsy
        val = point.get("value") if point.get("value") is not None else point.get("Value", 0)
        if raw_date is None:
            continue
        for fmt in _GDELT_DATE_FMTS:
            try:
                dates.append(datetime.strptime(str(raw_date).strip(), fmt))
                values.append(float(val or 0))
                break
            except (ValueError, TypeError):
                continue
    return dates, values

result = client.call_tool("intel_gdelt_timeline", {
    "query":     QUERY1,
    "query2":    QUERY2,
    "mode":      MODE,
    "timespan":  TIMESPAN,
    "smoothing": SMOOTHING,
}, timeout=90.0)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    series_list = result.get("series", [])
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    fig, ax = plt.subplots(figsize=(14, 5))
    plotted = 0

    for i, s in enumerate(series_list):
        label = s.get("query", f"Series {i+1}")
        data  = s.get("data", [])
        if not data:
            print(f"  ⚠  No data for '{label}' — try a broader timespan")
            continue

        # Multi-series modes (timelinesourcecountry / timelinelang) return
        # a list of {"series": "XX", "data": [{date, value}...]} objects
        if isinstance(data[0], dict) and "series" in data[0]:
            for sub in data[:6]:           # cap at 6 sub-series
                dates, values = _parse_series(sub.get("data", []))
                if dates:
                    ax.plot(dates, values, label=sub["series"], alpha=0.8)
                    plotted += 1
        else:
            dates, values = _parse_series(data)
            if dates:
                color = colors[i % len(colors)]
                ax.plot(dates, values, label=label, color=color, linewidth=2, alpha=0.9)
                ax.fill_between(dates, values, alpha=0.07, color=color)
                plotted += 1
            else:
                print(f"  ⚠  Could not parse dates for '{label}'. Raw sample: {data[:2]}")

    if plotted:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        fig.autofmt_xdate()
        ylabel = {
            "timelinevol":           "% of global news",
            "timelinevolraw":        "article count",
            "timelinetone":          "average tone",
            "timelinesourcecountry": "% of global news",
            "timelinelang":          "% of global news",
        }.get(MODE, "value")
        ax.set_ylabel(ylabel)
        ax.set_title(f"GDELT {MODE}  ·  {TIMESPAN}  ·  smooth={SMOOTHING}")
        ax.legend(loc="upper left")
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        plt.tight_layout()
        plt.show()
        total_pts = sum(len(s.get("data", [])) for s in series_list)
        print(f"{plotted} series  |  {total_pts} total data points")
    else:
        print("No plottable data — try a broader timespan or different query.")
        print(f"Raw series[0] sample: {(series_list or [{}])[0].get('data', [])[:3]}")

## 6. GDELT Timeline — Tone Comparison

In [ ]:
# GDELT Tone Timeline — positive values = positive coverage, negative = negative.
# Useful for tracking sentiment shifts around events.
# Requires _parse_series and _GDELT_DATE_FMTS defined in the cell above.

TONE_QUERY1   = "ukraine ceasefire"
TONE_QUERY2   = "russia sanctions"    # set to None to plot one topic
TONE_TIMESPAN = "2w"
TONE_SMOOTH   = 5

result = client.call_tool("intel_gdelt_timeline", {
    "query":     TONE_QUERY1,
    "query2":    TONE_QUERY2,
    "mode":      "timelinetone",
    "timespan":  TONE_TIMESPAN,
    "smoothing": TONE_SMOOTH,
}, timeout=90.0)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    series_list = result.get("series", [])
    colors = ["#1f77b4", "#d62728"]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")

    for i, s in enumerate(series_list):
        dates, values = _parse_series(s.get("data", []))
        if not dates:
            print(f"  ⚠  No data for '{s.get('query')}'. Raw sample: {s.get('data', [])[:2]}")
            continue
        color = colors[i % len(colors)]
        ax.plot(dates, values, label=s.get("query"), color=color, linewidth=2)
        ax.fill_between(dates, values, 0,
                        where=[v >= 0 for v in values],
                        interpolate=True, alpha=0.15, color="green")
        ax.fill_between(dates, values, 0,
                        where=[v < 0 for v in values],
                        interpolate=True, alpha=0.15, color="red")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    fig.autofmt_xdate()
    ax.set_ylabel("average tone  (+ positive  /  − negative)")
    ax.set_title(f"GDELT Tone Timeline  ·  {TONE_TIMESPAN}  ·  smooth={TONE_SMOOTH}")
    ax.legend(loc="upper left")
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Server Health — Data Source Status

In [ ]:
result = client.call_tool("intel_status")

cache   = result.get("cache", {})
problem = result.get("circuit_breakers", {})

print(f"Sources tracked : {result.get('sources_tracked', '?')}")
print(f"Cache entries   : {cache.get('entries', '?')}  |  {cache.get('size_mb', '?')} MB\n")

if problem:
    print(f"⚠️  {result.get('problem_sources', 0)} problem source(s):\n")
    for src, info in sorted(problem.items()):
        print(f"  [{info.get('status','?'):<10}] {src:<45} "
              f"failures={info.get('total_failures',0)}")
else:
    print("✅ All data sources healthy — no failures recorded\n")

print("\nData sources:")
for category, feeds in result.get("data_sources", {}).items():
    items = feeds if isinstance(feeds, list) else [feeds]
    for f in items:
        print(f"  • {f}")

## 8. Custom Tool Call
Call any tool by name with arbitrary arguments.

In [ ]:
# ---- Customize these ----
TOOL_NAME = "intel_gdelt_search"
ARGUMENTS = {"query": "taiwan strait", "mode": "artlist", "limit": 10}
# -------------------------

result = client.call_tool(TOOL_NAME, ARGUMENTS, timeout=30.0)
display(JSON(result))

## 9. Disconnect

In [ ]:
client.disconnect()